In [1]:
import re
import json
import pandas as pd
from spacy import displacy
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

# ============================================================
# 1. LOAD SEMUA MODEL
# ============================================================
print("Loading models...")

# Model POS Tagger
pos_tokenizer = AutoTokenizer.from_pretrained("w11wo/indonesian-roberta-base-posp-tagger")
pos_model     = AutoModelForTokenClassification.from_pretrained("w11wo/indonesian-roberta-base-posp-tagger")
pos_pipeline  = pipeline("token-classification", model=pos_model, tokenizer=pos_tokenizer,
                         aggregation_strategy="simple")

# Model NER (aggregated untuk displaCy)
ner_tokenizer = AutoTokenizer.from_pretrained("cahya/bert-base-indonesian-NER")
ner_model     = AutoModelForTokenClassification.from_pretrained("cahya/bert-base-indonesian-NER")
ner_pipeline  = pipeline("ner", model=ner_model, tokenizer=ner_tokenizer,
                         aggregation_strategy="simple")

# Model NER (raw untuk IOB tags)
ner_pipeline_raw = pipeline("ner", model=ner_model, tokenizer=ner_tokenizer)

print("Semua model berhasil dimuat!\n") 

# ============================================================
# 2. FUNGSI PREPROCESSING
# ============================================================
def clean_text(text):
    text = re.sub(r'\d+\s+(?:[A-Za-z]\s+){2,}[A-Za-z]+', '', text)
    text = re.sub(r'Kumpulan Dongeng nusantara\s+\d+', '', text)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        words = line.strip().split()
        if len(words) > 3 or len(words) == 0:
            cleaned_lines.append(line)
    text = '\n'.join(cleaned_lines)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def chunk_text(text, max_length=400):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current = [], ""
    for sent in sentences:
        if len(current) + len(sent) < max_length:
            current += " " + sent
        else:
            if current:
                chunks.append(current.strip())
            current = sent
    if current:
        chunks.append(current.strip())
    return chunks

# ============================================================
# 3. FUNGSI TOKENISASI (word-level)
# ============================================================
def tokenize(text):
    """Tokenisasi sederhana berbasis spasi dan tanda baca"""
    tokens = re.findall(r'\w+|[^\w\s]', text)
    return tokens

# ============================================================
# 4. FUNGSI NOUN PHRASE GROUPING DARI HASIL POS TAGS
# ============================================================
# Tag noun di Indonesian POSP: NN (common noun), NNP (proper noun), NNS (plural noun)
NOUN_TAGS = {"NN", "NNP", "NNS", "NNPS"}

def extract_noun_phrases(pos_results):
    """
    Mengelompokkan token berurutan yang memiliki POS tag noun
    menjadi noun phrases
    """
    noun_phrases = []
    current_np = []

    for token in pos_results:
        tag = token["entity_group"]
        word = token["word"]

        if tag in NOUN_TAGS:
            current_np.append(word)
        else:
            if current_np:
                noun_phrases.append(" ".join(current_np))
                current_np = []

    if current_np:
        noun_phrases.append(" ".join(current_np))

    return noun_phrases

# ============================================================
# 5. FUNGSI KONVERSI KE FORMAT DISPLACY
# ============================================================
def to_displacy_format(text, hf_entities):
    ents = []
    for ent in hf_entities:
        ents.append({
            "start": ent["start"],
            "end":   ent["end"],
            "label": ent["entity_group"]
        })
    return {"text": text, "ents": ents, "title": None}

# ============================================================
# 6. LOAD DATASET
# ============================================================
with open(r"C:\Semester 4\NLP\Pembelajaran Bahasa Mendalam\Tugas NER\data\dataset_ner_200.json", encoding="utf-8") as f:
    data = json.load(f)

print(f"Dataset berhasil dimuat: {len(data)} dokumen\n")

# ============================================================
# 7. PIPELINE UTAMA — PROSES SEMUA DOKUMEN
# ============================================================
all_tokens       = []   # hasil tokenisasi
all_pos_results  = []   # hasil POS tagging
all_noun_phrases = []   # hasil noun phrase grouping
all_iob_rows     = []   # hasil IOB tags
all_aggregated   = []   # hasil NER agregasi (untuk displaCy)

print("Memproses semua dokumen...")

for doc_idx, item in enumerate(data):

    # --- Step 1: Cleaning ---
    teks = clean_text(item["isi"])

    # --- Step 2: Chunking (potong teks < 512 token) ---
    chunks = chunk_text(teks)

    for chunk_idx, chunk in enumerate(chunks):

        # --- Step 3: Tokenisasi ---
        tokens = tokenize(chunk)
        all_tokens.append({
            "doc_idx":   doc_idx,
            "chunk_idx": chunk_idx,
            "tokens":    tokens,
            "jumlah":    len(tokens)
        })

        # --- Step 4: POS Tagging ---
        pos_results = pos_pipeline(chunk)
        for pos in pos_results:
            all_pos_results.append({
                "doc_idx":   doc_idx,
                "chunk_idx": chunk_idx,
                "word":      pos["word"],
                "pos_tag":   pos["entity_group"],
                "score":     round(pos["score"], 4)
            })

        # --- Step 5: Noun Phrase Grouping ---
        noun_phrases = extract_noun_phrases(pos_results)
        for np_text in noun_phrases:
            all_noun_phrases.append({
                "doc_idx":     doc_idx,
                "chunk_idx":   chunk_idx,
                "noun_phrase": np_text
            })

        # --- Step 6: IOB Tags (raw NER) ---
        raw_entities = ner_pipeline_raw(chunk)
        for ent in raw_entities:
            all_iob_rows.append({
                "doc_idx":   doc_idx,
                "chunk_idx": chunk_idx,
                "word":      ent["word"],
                "iob_tag":   ent["entity"],
                "score":     round(ent["score"], 4)
            })

        # --- Step 7: Extract Named Entities (aggregated) ---
        agg_entities = ner_pipeline(chunk)
        all_aggregated.append({
            "doc_idx":  doc_idx,
            "chunk":    chunk,
            "entities": agg_entities
        })

print(f"Selesai! Total chunk diproses: {len(all_aggregated)}\n")

# ============================================================
# 8. KONVERSI KE DATAFRAME
# ============================================================
df_tokens  = pd.DataFrame(all_tokens)
df_pos     = pd.DataFrame(all_pos_results)
df_np      = pd.DataFrame(all_noun_phrases)
df_iob     = pd.DataFrame(all_iob_rows)

# Rekap entitas dari hasil agregasi
rekap = []
for item in all_aggregated:
    for ent in item["entities"]:
        rekap.append({
            "doc_idx": item["doc_idx"],
            "entity":  ent["word"],
            "label":   ent["entity_group"],
            "score":   round(ent["score"], 4)
        })
df_rekap = pd.DataFrame(rekap)

# ============================================================
# 9. TAMPILKAN HASIL PER LANGKAH (DOKUMEN 0)
# ============================================================

print("=" * 60)
print("LANGKAH 1 — TOKENISASI (Dokumen 0, Chunk 0)")
print("=" * 60)
doc0_tokens = df_tokens[(df_tokens["doc_idx"] == 0) & (df_tokens["chunk_idx"] == 0)]
print(f"Jumlah token: {doc0_tokens['jumlah'].values[0]}")
print(f"Token: {doc0_tokens['tokens'].values[0][:20]} ...")

print("\n" + "=" * 60)
print("LANGKAH 2 — POS TAGGING (Dokumen 0)")
print("=" * 60)
df_pos_doc0 = df_pos[df_pos["doc_idx"] == 0].reset_index(drop=True)
display(df_pos_doc0.head(30))

print("\n" + "=" * 60)
print("LANGKAH 3 — NOUN PHRASE GROUPING (Dokumen 0)")
print("=" * 60)
df_np_doc0 = df_np[df_np["doc_idx"] == 0].reset_index(drop=True)
display(df_np_doc0.head(20))

print("\n" + "=" * 60)
print("LANGKAH 4 — IOB TAGS (Dokumen 0)")
print("=" * 60)
df_iob_doc0 = df_iob[(df_iob["doc_idx"] == 0) & (df_iob["iob_tag"] != "O")].reset_index(drop=True)
display(df_iob_doc0.head(30))

print("\n" + "=" * 60)
print("LANGKAH 5 — NAMED ENTITIES + VISUALISASI DISPLACY (Dokumen 0)")
print("=" * 60)

colors = {
    "PER": "#f9a825",
    "LOC": "#42a5f5",
    "ORG": "#66bb6a",
}

doc0_chunks = [x for x in all_aggregated if x["doc_idx"] == 0]
for i, item in enumerate(doc0_chunks):
    doc_data = to_displacy_format(item["chunk"], item["entities"])
    print(f"\n--- Chunk {i+1} ---")
    displacy.render(doc_data, style="ent", manual=True, jupyter=True,
                    options={"colors": colors})

# ============================================================
# 10. ANALISIS TAMBAHAN — NNP vs NER
# ============================================================
print("\n" + "=" * 60)
print("ANALISIS — DISTRIBUSI POS TAGS (semua dokumen)")
print("=" * 60)
display(df_pos["pos_tag"].value_counts().reset_index()
        .rename(columns={"index": "pos_tag", "pos_tag": "frekuensi"}))

print("\n" + "=" * 60)
print("ANALISIS — FREKUENSI LABEL NER (semua dokumen)")
print("=" * 60)
display(df_rekap["label"].value_counts())

print("\n" + "=" * 60)
print("ANALISIS — ENTITAS PALING SERING MUNCUL (Top 20)")
print("=" * 60)
display(df_rekap.groupby(["entity", "label"]).size()
        .reset_index(name="frekuensi")
        .sort_values("frekuensi", ascending=False)
        .head(20))

print("\n" + "=" * 60)
print("ANALISIS — PERBANDINGAN NOUN PHRASES vs NAMED ENTITIES")
print("=" * 60)
total_np  = len(df_np)
total_ner = len(df_rekap)
np_set    = set(df_np["noun_phrase"].str.lower())
ner_set   = set(df_rekap["entity"].str.lower())
overlap   = np_set & ner_set

print(f"Total Noun Phrases ditemukan : {total_np}")
print(f"Total Named Entities ditemukan: {total_ner}")
print(f"Overlap (NP yang juga jadi NER): {len(overlap)}")
print(f"Coverage: {round(len(overlap)/len(ner_set)*100, 2)}% NER tercakup dalam Noun Phrases")

Loading models...


Device set to use cpu
Some weights of the model checkpoint at cahya/bert-base-indonesian-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Semua model berhasil dimuat!

Dataset berhasil dimuat: 200 dokumen

Memproses semua dokumen...


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Selesai! Total chunk diproses: 2858

LANGKAH 1 — TOKENISASI (Dokumen 0, Chunk 0)
Jumlah token: 64
Token: ['Lilo', 'adalah', 'nyamuk', 'jantan', 'kecil', 'yang', 'cerdas', '.', 'Dia', 'dan', 'keluarganya', 'tinggal', 'di', 'Kolam', 'Tamansari', 'Kerajaan', 'Ardiloka', '.', 'Kerajaan', 'Ardiloka'] ...

LANGKAH 2 — POS TAGGING (Dokumen 0)


...


ANALISIS — FREKUENSI LABEL NER (semua dokumen)


label
PER    7183
LOC     964
PRD     852
CRD     707
ORG     603
GPE     574
NOR     162
WOA     127
QTY     123
REG      98
EVT      98
ORD      57
DAT      27
TIM      11
FAC       8
LAN       5
LAW       3
Name: count, dtype: int64


ANALISIS — ENTITAS PALING SERING MUNCUL (Top 20)


,entity,label,frekuensi
1527,kedua,CRD,142
2846,satu,CRD,109
405,aji,PER,107
1029,dua,CRD,105
1682,kiki,PER,78
993,dimas,PER,68
1340,jaka bagus,PER,65
3248,tuhan,REG,61
1339,jaka,PER,58
2659,raja,PER,55



ANALISIS — PERBANDINGAN NOUN PHRASES vs NAMED ENTITIES
Total Noun Phrases ditemukan : 42240
Total Named Entities ditemukan: 11602
Overlap (NP yang juga jadi NER): 50
Coverage: 1.71% NER tercakup dalam Noun Phrases
